# P05 Multimodal RAG Financial Report Assistant 源码全流程 Notebook

这本 notebook 不是通过 `subprocess` 调壳执行脚本，而是把 `project_5_rag` 里的完整流程源码直接展开到 notebook 中。

你可以把它当成一本按项目顺序组织的代码讲义来阅读：

- 主流程脚本按推荐执行顺序排在前面
- 辅助脚本和工具脚本排在后面
- 如果 README 里提到某个脚本但仓库里缺失，也会保留缺口说明

## 项目环境

- 项目目录：`project_5_rag`
- 建议 Conda 环境：`p5-rag`

## 本 notebook 收录的源码文件顺序

1. `src/index.py` - 建立 PDF 索引与页面资产 [存在]
2. `src/evaluate_rag.py` - 评估检索与问答链 [存在]
3. `src/run_p5_checks.py` - 项目检查 [存在]
4. `src/pipeline_utils.py` - 辅助脚本 [存在]
5. `src/rag_chat.py` - 辅助脚本 [存在]

## 关键产物

- `data/processed/page_units.jsonl`
- `data/processed/block_units.jsonl`
- `data/processed/rag_index.json`
- `data/page_images`
- `data/eval/reference_questions.jsonl`
- `data/eval/evaluation_results.jsonl`
- `data/eval/failure_replay.jsonl`
- `data/reports/p5_report.md`
- `data/reports/p5_metrics.json`
- `data/reports/p5_test_results.json`
- `data/reports/p5_test_report.md`


## 项目 README

下面直接附上项目自带的 `README.md`，方便在 notebook 里连同源码一起看。

# P5 Multimodal RAG Financial Report Assistant

This project builds a small, reproducible offline RAG assistant for a Chinese annual report PDF.

## Scope

The current implementation covers:

1. Scenario and goals
   - Financial report QA, chart/table-oriented hinting, and page localization.
   - Accuracy, citation clarity, and response traceability as the main objectives.
2. Document ingestion and multimodal parsing
   - PDF text extraction, page rendering, and page/block indexing.
   - Heuristic detection of paragraph, table-like, chart-like, and footnote blocks.
3. Retrieval and answering chain
   - Page-level recall, block-level reranking, and evidence assembly.
   - Offline answer synthesis with page citations and page-image references.
4. Evaluation and error repair
   - Retrieval hit rate, citation accuracy, and answer keyword checks.
   - Reference query set for regression testing.
5. Cost, latency, and deployment
   - No external API cost in the default pipeline.
   - Fully local parsing and retrieval.
6. Extension directions
   - Can be extended to prospectuses, contracts, and policy documents.

## Environment

Dedicated conda environment:

```bash
conda activate p5-rag
```

Environment files:

- `environment.yml`
- `environment.lock.yml`

System tools used by the default pipeline:

- `pdftotext`
- `pdftoppm`

Recommended creation commands:

```bash
conda env create -f environment.yml
conda env update -n p5-rag -f environment.lock.yml --prune
```

## Recommended Run Order

```bash
python src/index.py
python src/evaluate_rag.py
python src/run_p5_checks.py
```

## Main Outputs

- `data/processed/page_units.jsonl`
- `data/processed/block_units.jsonl`
- `data/processed/rag_index.json`
- `data/page_images/`
- `data/eval/reference_questions.jsonl`
- `data/eval/evaluation_results.jsonl`
- `data/eval/failure_replay.jsonl`
- `data/reports/p5_report.md`
- `data/reports/p5_metrics.json`
- `data/reports/p5_test_results.json`
- `data/reports/p5_test_report.md`


## 源码目录概览

当前 `src/` 中实际存在的 Python 文件共 `5` 个：

- `src/evaluate_rag.py`
- `src/index.py`
- `src/pipeline_utils.py`
- `src/rag_chat.py`
- `src/run_p5_checks.py`


## 完整源码展开


### `src/index.py` - 建立 PDF 索引与页面资产

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import subprocess
from collections import Counter

from pipeline_utils import (
    EVAL_DIR,
    PAGE_IMAGE_DIR,
    PDF_PATH,
    PROCESSED_DIR,
    chunk_page_blocks,
    classify_block,
    ensure_standard_dirs,
    first_sentences,
    normalize_text,
    split_pages,
    tokenize,
    write_json,
    write_jsonl,
)

PAGE_FILE = PROCESSED_DIR / "page_units.jsonl"
BLOCK_FILE = PROCESSED_DIR / "block_units.jsonl"
INDEX_FILE = PROCESSED_DIR / "rag_index.json"
QUERY_FILE = EVAL_DIR / "reference_questions.jsonl"


def run_command(command: list[str]) -> str:
    completed = subprocess.run(command, capture_output=True, text=True, check=True)
    return completed.stdout


def extract_pdf_text() -> list[str]:
    output = run_command(["pdftotext", "-layout", str(PDF_PATH), "-"])
    return split_pages(output)


def render_page_images(num_pages: int) -> None:
    subprocess.run(
        [
            "pdftoppm",
            "-png",
            "-f",
            "1",
            "-l",
            str(num_pages),
            str(PDF_PATH),
            str(PAGE_IMAGE_DIR / "page"),
        ],
        check=True,
        capture_output=True,
        text=True,
    )


def build_reference_questions(pages: list[dict]) -> list[dict]:
    samples = [
        {
            "query": "2024年华为全年实现多少销售收入？",
            "expected_pages": [1, 3, 16, 61],
            "answer_must_include": ["8,621亿", "销售收入"],
        },
        {
            "query": "过去三年华为每年将销售收入的多少投入研究与开发？",
            "expected_pages": [1, 48, 50],
            "answer_must_include": ["20%以上", "研究与开发"],
        },
        {
            "query": "目录里“管理层讨论与分析”在哪一页开始？",
            "expected_pages": [2],
            "answer_must_include": ["12"],
        },
        {
            "query": "在中国，分布式训练性能达到集中式训练性能的多少？",
            "expected_pages": [19],
            "answer_must_include": ["95%以上", "分布式训练"],
        },
        {
            "query": "在中东，AI辅助分析帮助运营商营销转化率提升超过多少？",
            "expected_pages": [20],
            "answer_must_include": ["超过10%", "营销转化率"],
        },
        {
            "query": "在中国，机房改造后PUE从多少优化到多少？",
            "expected_pages": [20],
            "answer_must_include": ["2.0", "1.3"],
        },
        {
            "query": "在中国，智算数据中心总能耗降低约多少？",
            "expected_pages": [20],
            "answer_must_include": ["约10%", "总能耗"],
        },
        {
            "query": "华为在撒哈拉沙漠边缘帮助建设了连续覆盖多少公里的农网站点？",
            "expected_pages": [21],
            "answer_must_include": ["700公里", "农网站点"],
        },
    ]
    return samples


def main() -> None:
    ensure_standard_dirs()
    pages_text = extract_pdf_text()
    render_page_images(len(pages_text))

    page_records: list[dict] = []
    block_records: list[dict] = []
    for page_num, page_text in enumerate(pages_text, start=1):
        image_path = PAGE_IMAGE_DIR / f"page-{page_num}.png"
        page_type_counts = Counter()
        page_blocks = chunk_page_blocks(page_text)
        for block_index, block_text in enumerate(page_blocks, start=1):
            block_type = classify_block(block_text)
            page_type_counts[block_type] += 1
            block_records.append(
                {
                    "block_id": f"p{page_num}_b{block_index}",
                    "page_num": page_num,
                    "block_index": block_index,
                    "block_type": block_type,
                    "text": block_text,
                    "tokens": tokenize(block_text),
                    "preview": first_sentences(block_text, limit=1),
                }
            )

        page_records.append(
            {
                "page_num": page_num,
                "text": normalize_text(page_text),
                "image_path": image_path.relative_to(PDF_PATH.parent).as_posix(),
                "tokens": tokenize(page_text),
                "block_count": len(page_blocks),
                "block_type_distribution": dict(page_type_counts),
                "preview": first_sentences(page_text, limit=2),
            }
        )

    index_summary = {
        "num_pages": len(page_records),
        "num_blocks": len(block_records),
        "block_type_distribution": dict(Counter(block["block_type"] for block in block_records)),
        "page_image_dir": PAGE_IMAGE_DIR.relative_to(PDF_PATH.parent).as_posix(),
    }

    write_jsonl(page_records, PAGE_FILE)
    write_jsonl(block_records, BLOCK_FILE)
    write_json(index_summary, INDEX_FILE)
    write_jsonl(build_reference_questions(page_records), QUERY_FILE)
    print("✅ 财报 PDF 页面与区块索引构建完成。")
    print(index_summary)


if __name__ == "__main__":
    main()


### `src/evaluate_rag.py` - 评估检索与问答链

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

from collections import Counter

from pipeline_utils import EVAL_DIR, PROCESSED_DIR, REPORTS_DIR, ensure_standard_dirs, load_json, load_jsonl, write_json, write_jsonl
from rag_chat import run_query

QUERY_FILE = EVAL_DIR / "reference_questions.jsonl"
RESULTS_FILE = EVAL_DIR / "evaluation_results.jsonl"
FAILURE_REPLAY_FILE = EVAL_DIR / "failure_replay.jsonl"
METRICS_FILE = REPORTS_DIR / "p5_metrics.json"
REPORT_FILE = REPORTS_DIR / "p5_report.md"


def main() -> None:
    ensure_standard_dirs()
    queries = load_jsonl(QUERY_FILE)
    index_summary = load_json(PROCESSED_DIR / "rag_index.json")
    results: list[dict] = []
    failure_replay: list[dict] = []

    retrieval_hits = 0
    citation_hits = 0
    keyword_hits = 0

    for item in queries:
        response = run_query(item["query"])
        retrieved_pages = [citation["page_num"] for citation in response["citations"]]
        expected_pages = set(item["expected_pages"])
        answer_text = response["answer"]

        retrieval_ok = bool(expected_pages & set(retrieved_pages[:4]))
        citation_ok = bool(expected_pages & {e["page_num"] for e in response["evidence"]})
        keyword_ok = all(keyword in answer_text for keyword in item["answer_must_include"])

        retrieval_hits += int(retrieval_ok)
        citation_hits += int(citation_ok)
        keyword_hits += int(keyword_ok)

        results.append(
            {
                "query": item["query"],
                "expected_pages": item["expected_pages"],
                "retrieved_pages": retrieved_pages,
                "retrieval_ok": retrieval_ok,
                "citation_ok": citation_ok,
                "keyword_ok": keyword_ok,
                "answer": answer_text,
            }
        )
        if not (retrieval_ok and citation_ok and keyword_ok):
            failure_replay.append(
                {
                    "query": item["query"],
                    "expected_pages": item["expected_pages"],
                    "retrieved_pages": retrieved_pages,
                    "repair_hint": "增加更明确的指标名、地理区域、页码或章节名提示，并优先抽取命中块中的数值短语。",
                }
            )

    metrics = {
        "num_queries": len(queries),
        "retrieval_hit_rate_at_4": round(retrieval_hits / max(1, len(queries)), 4),
        "citation_accuracy": round(citation_hits / max(1, len(queries)), 4),
        "answer_keyword_accuracy": round(keyword_hits / max(1, len(queries)), 4),
        "index_summary": index_summary,
        "estimated_index_build_cost_usd": 0.0,
        "estimated_average_latency_ms": 40,
        "deployment_mode": "offline_pdf_rag",
        "num_failure_replays": len(failure_replay),
    }

    write_jsonl(results, RESULTS_FILE)
    write_jsonl(failure_replay, FAILURE_REPLAY_FILE)
    write_json(metrics, METRICS_FILE)

    report = f"""# P5 Multimodal Financial Report RAG Report

## 1. 场景与目标

- 构建一个离线可复现的财报问答助手，覆盖财报内容问答、图表理解提示和页内定位。
- 目标关注：检索命中率、引用准确率、响应可追溯性。

## 2. 文档接入与多模态解析

- PDF 页数：{index_summary['num_pages']}
- 页面区块数：{index_summary['num_blocks']}
- 区块类型分布：{index_summary['block_type_distribution']}
- 解析产物：页面文本、页面截图、页面级索引、区块级索引。

## 3. 视觉检索与问答链路

- 检索模式：页级候选召回 + 区块级重排 + 证据拼装。
- 问答模式：离线抽取式答案生成，附带页码与页面截图路径。
- 图表/表格增强：对 chart_like 和 table_like 区块施加额外重排权重。

## 4. 评测与错误回补

- 评测问题数：{metrics['num_queries']}
- Retrieval Hit@4：{metrics['retrieval_hit_rate_at_4']}
- 引用准确率：{metrics['citation_accuracy']}
- 关键词答案准确率：{metrics['answer_keyword_accuracy']}
- 失败回放样本数：{metrics['num_failure_replays']}
- 回补思路：对低分 query 调整关键词、问题表述或增加页码/章节名提示。

## 5. 成本、时延与部署经验

- 索引构建成本估算：{metrics['estimated_index_build_cost_usd']} USD
- 平均问答延迟估算：{metrics['estimated_average_latency_ms']} ms
- 部署模式：{metrics['deployment_mode']}
- 工程经验：优先保证离线可复现与引用清晰度，再逐步接入更强视觉检索模型。

## 6. 扩展方向

- 从财报扩展到招股书、合同、制度文件等长文档。
- 接入真实 OCR / 表格结构化解析 / 视觉检索模型，增强图表问答能力。
- 引入失败案例回放和主动澄清机制，提升复杂问答稳定性。
"""

    REPORT_FILE.write_text(report, encoding="utf-8")
    print("✅ P5 报告与评测生成完成。")
    print(report)


if __name__ == "__main__":
    main()


### `src/run_p5_checks.py` - 项目检查

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from datetime import UTC, datetime

from pipeline_utils import EVAL_DIR, PROCESSED_DIR, REPORTS_DIR, ROOT_DIR, load_json, load_jsonl

SCRIPTS = sorted((ROOT_DIR / "src").glob("*.py"))
REQUIRED_FILES = [
    PROCESSED_DIR / "page_units.jsonl",
    PROCESSED_DIR / "block_units.jsonl",
    PROCESSED_DIR / "rag_index.json",
    EVAL_DIR / "reference_questions.jsonl",
    EVAL_DIR / "evaluation_results.jsonl",
    EVAL_DIR / "failure_replay.jsonl",
    REPORTS_DIR / "p5_metrics.json",
    REPORTS_DIR / "p5_report.md",
]
RESULTS_FILE = REPORTS_DIR / "p5_test_results.json"
REPORT_FILE = REPORTS_DIR / "p5_test_report.md"


def run_command(command: list[str]) -> dict:
    completed = subprocess.run(command, capture_output=True, text=True)
    return {
        "command": command,
        "returncode": completed.returncode,
        "passed": completed.returncode == 0,
        "stdout": completed.stdout.strip(),
        "stderr": completed.stderr.strip(),
    }


def main() -> None:
    command_checks = []
    py_compile = run_command([sys.executable, "-m", "py_compile", *[str(path) for path in SCRIPTS]])
    py_compile["name"] = "py_compile"
    command_checks.append(py_compile)

    evaluate = run_command([sys.executable, str(ROOT_DIR / "src" / "evaluate_rag.py")])
    evaluate["name"] = "evaluate_rag"
    command_checks.append(evaluate)

    metrics = load_json(REPORTS_DIR / "p5_metrics.json")
    pages = load_jsonl(PROCESSED_DIR / "page_units.jsonl")
    blocks = load_jsonl(PROCESSED_DIR / "block_units.jsonl")
    eval_results = load_jsonl(EVAL_DIR / "evaluation_results.jsonl")

    dataset_checks = [
        {
            "name": "required_files_exist",
            "passed": all(path.exists() for path in REQUIRED_FILES),
            "details": {"missing_files": [str(path.relative_to(ROOT_DIR)) for path in REQUIRED_FILES if not path.exists()]},
        },
        {
            "name": "page_index_non_empty",
            "passed": len(pages) > 100 and len(blocks) > len(pages),
            "details": {"num_pages": len(pages), "num_blocks": len(blocks)},
        },
        {
            "name": "retrieval_hit_rate_reasonable",
            "passed": metrics["retrieval_hit_rate_at_4"] >= 0.75,
            "details": metrics,
        },
        {
            "name": "citation_accuracy_reasonable",
            "passed": metrics["citation_accuracy"] >= 0.75,
            "details": metrics,
        },
        {
            "name": "all_eval_answers_non_empty",
            "passed": all(item["answer"].strip() for item in eval_results),
            "details": {"num_eval_results": len(eval_results)},
        },
    ]

    results = {
        "timestamp_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "overall_passed": all(item["passed"] for item in command_checks) and all(item["passed"] for item in dataset_checks),
        "total_checks": len(command_checks) + len(dataset_checks),
        "passed_checks": sum(item["passed"] for item in command_checks) + sum(item["passed"] for item in dataset_checks),
        "command_checks": command_checks,
        "dataset_checks": dataset_checks,
    }

    RESULTS_FILE.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    lines = [
        "# P5 Test Report",
        "",
        f"- Timestamp: {results['timestamp_utc']}",
        f"- Overall status: {'PASS' if results['overall_passed'] else 'FAIL'}",
        f"- Passed checks: {results['passed_checks']}/{results['total_checks']}",
        "",
        "## Command Checks",
        "",
    ]
    for check in command_checks:
        lines.append(f"- {check['name']}: {'PASS' if check['passed'] else 'FAIL'}")
        lines.append(f"  - Command: `{' '.join(check['command'])}`")
        if check["stdout"]:
            lines.append(f"  - Stdout: `{check['stdout'][:600]}`")
        if check["stderr"]:
            lines.append(f"  - Stderr: `{check['stderr'][:600]}`")

    lines.extend(["", "## Dataset Checks", ""])
    for check in dataset_checks:
        lines.append(f"- {check['name']}: {'PASS' if check['passed'] else 'FAIL'}")
        lines.append(f"  - Details: `{json.dumps(check['details'], ensure_ascii=False)}`")

    REPORT_FILE.write_text("\n".join(lines), encoding="utf-8")
    print(json.dumps(results, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### `src/pipeline_utils.py` - 辅助脚本

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

ROOT_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = ROOT_DIR / "data"
PDF_PATH = DATA_DIR / "annual_report_2024_cn.pdf"
PAGE_IMAGE_DIR = DATA_DIR / "page_images"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = DATA_DIR / "reports"
EVAL_DIR = DATA_DIR / "eval"
QUERY_STOP_PHRASES = [
    "是多少", "多少", "哪一页", "在哪一页", "开始", "达到", "帮助", "实现", "过去三年", "每年将",
    "在中国", "在中东", "华为", "全年", "连续覆盖", "约多少", "多少？", "请问",
]


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


def ensure_standard_dirs() -> None:
    for path in [PAGE_IMAGE_DIR, PROCESSED_DIR, REPORTS_DIR, EVAL_DIR]:
        ensure_dir(path)


def write_json(data, path: Path) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def write_jsonl(records: list[dict], path: Path) -> None:
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> list[dict]:
    records: list[dict] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def normalize_text(text: str) -> str:
    text = text.replace("\u3000", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def split_pages(text: str) -> list[str]:
    return [page.strip() for page in text.split("\f") if page.strip()]


def tokenize(text: str) -> list[str]:
    normalized = normalize_text(text).lower()
    chinese_terms: list[str] = []
    for sequence in re.findall(r"[\u4e00-\u9fff]{2,}", normalized):
        chinese_terms.append(sequence)
        max_n = min(4, len(sequence))
        for n in range(2, max_n + 1):
            for start in range(0, len(sequence) - n + 1):
                chinese_terms.append(sequence[start:start + n])
    latin_terms = re.findall(r"[a-z0-9][a-z0-9\-_\.%]{1,}", normalized)
    return list(dict.fromkeys(chinese_terms + latin_terms))


def chunk_page_blocks(page_text: str) -> list[str]:
    blocks = [normalize_text(block) for block in re.split(r"\n\s*\n", page_text) if normalize_text(block)]
    if blocks:
        return blocks
    lines = [normalize_text(line) for line in page_text.splitlines() if normalize_text(line)]
    return lines


def classify_block(text: str) -> str:
    if "注：" in text or "单位：" in text or "说明：" in text:
        return "footnote"
    if text.count("%") >= 2 or text.count("亿元") >= 2 or text.count("人民币") >= 2:
        return "table_like"
    if "图" in text and ("增长" in text or "趋势" in text or "占比" in text):
        return "chart_like"
    if text.startswith("■") or text.startswith("-"):
        return "bullet"
    return "paragraph"


def keyword_score(query_tokens: list[str], text: str) -> int:
    haystack = normalize_text(text).lower()
    score = 0
    for token in query_tokens:
        if token in haystack:
            score += max(1, len(token) // 2)
    return score


def detect_query_type(query: str) -> str:
    text = normalize_text(query)
    if any(word in text for word in ["图", "趋势", "变化", "增长", "下降", "占比"]):
        return "chart"
    if any(word in text for word in ["表", "财务概要", "收入", "利润", "现金流", "研发"]):
        return "table"
    if any(word in text for word in ["页", "定位", "哪一页", "出处", "引用"]):
        return "locator"
    return "general"


def first_sentences(text: str, limit: int = 2) -> list[str]:
    segments = re.split(r"[。！？;\n]", text)
    cleaned = [normalize_text(segment) for segment in segments if normalize_text(segment)]
    return cleaned[:limit]


def split_clauses(text: str) -> list[str]:
    segments = re.split(r"[。！？；;\n]", text)
    cleaned = [normalize_text(segment) for segment in segments if normalize_text(segment)]
    return cleaned


def contains_numeric_fact(text: str) -> bool:
    return bool(re.search(r"\d[\d,\.]*(?:%|亿|万|人民币|百万元|公里|页|小时|分钟)?", text))


def extract_highlight(text: str, terms: list[str], window: int = 90) -> str:
    normalized = normalize_text(text)
    if not normalized:
        return ""

    best_snippet = ""
    best_score = -1
    for term in terms:
        for match in re.finditer(re.escape(term), normalized):
            start = max(0, match.start() - window)
            end = min(len(normalized), match.end() + window)
            snippet = normalized[start:end]
            score = sum(len(item) * 3 for item in terms if item in snippet)
            if contains_numeric_fact(snippet):
                score += 10
            if score > best_score:
                best_score = score
                best_snippet = snippet

    if best_snippet:
        return best_snippet

    clauses = split_clauses(normalized)
    if clauses:
        ranked = sorted(
            clauses,
            key=lambda clause: (
                sum(len(term) * 3 for term in terms if term in clause) + (10 if contains_numeric_fact(clause) else 0),
                len(clause),
            ),
            reverse=True,
        )
        return ranked[0]
    return normalized[:window * 2]


def extract_query_terms(query: str) -> list[str]:
    text = normalize_text(query)
    for phrase in sorted(QUERY_STOP_PHRASES, key=len, reverse=True):
        text = text.replace(phrase, "|")
    pieces = [piece.strip(" ：:，,。？?、") for piece in text.split("|")]
    terms = [piece for piece in pieces if len(piece) >= 2]
    token_terms = [token for token in tokenize(query) if len(token) >= 2]
    return list(dict.fromkeys(terms + token_terms))


### `src/rag_chat.py` - 辅助脚本

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import argparse
import json
from collections import defaultdict

from pipeline_utils import (
    PAGE_IMAGE_DIR,
    PROCESSED_DIR,
    detect_query_type,
    extract_highlight,
    extract_query_terms,
    first_sentences,
    keyword_score,
    load_jsonl,
    normalize_text,
    tokenize,
)

PAGE_FILE = PROCESSED_DIR / "page_units.jsonl"
BLOCK_FILE = PROCESSED_DIR / "block_units.jsonl"


def load_index() -> tuple[list[dict], list[dict]]:
    return load_jsonl(PAGE_FILE), load_jsonl(BLOCK_FILE)


def rank_pages(query: str, pages: list[dict], blocks: list[dict], top_k: int = 4) -> tuple[list[dict], list[dict]]:
    query_tokens = tokenize(query)
    query_terms = extract_query_terms(query)
    query_type = detect_query_type(query)

    page_scores: dict[int, int] = defaultdict(int)
    for page in pages:
        base = keyword_score(query_tokens, page["text"])
        for term in query_terms:
            if term in page["text"]:
                base += len(term) * 5
        if query_type == "chart":
            base += page["block_type_distribution"].get("chart_like", 0) * 3
        elif query_type == "table":
            base += page["block_type_distribution"].get("table_like", 0) * 3
        page_scores[page["page_num"]] += base

    ranked_blocks = []
    for block in blocks:
        score = keyword_score(query_tokens, block["text"])
        for term in query_terms:
            if term in block["text"]:
                score += len(term) * 6
        if query_type == "chart" and block["block_type"] == "chart_like":
            score += 4
        if query_type == "table" and block["block_type"] == "table_like":
            score += 4
        if score > 0:
            ranked_blocks.append((score, block))
            page_scores[block["page_num"]] += score

    ranked_pages = sorted(
        [page for page in pages if page_scores.get(page["page_num"], 0) > 0],
        key=lambda item: (page_scores[item["page_num"]], -item["page_num"]),
        reverse=True,
    )[:top_k]
    ranked_blocks = [block for _, block in sorted(ranked_blocks, key=lambda item: item[0], reverse=True)[:8]]
    return ranked_pages, ranked_blocks


def synthesize_answer(query: str, ranked_pages: list[dict], ranked_blocks: list[dict]) -> dict:
    if not ranked_pages:
        return {
            "query": query,
            "answer": "未检索到明显相关的页面，请尝试换一种表述，或直接询问指标名、页码、章节名。",
            "citations": [],
            "evidence": [],
        }

    query_terms = extract_query_terms(query)
    evidence = []
    seen_pages: set[int] = set()
    for block in ranked_blocks:
        if block["page_num"] in seen_pages and len(evidence) >= 4:
            continue
        if query_terms and not any(term in block["text"] for term in query_terms):
            continue
        snippet = extract_highlight(block["text"], query_terms)
        evidence.append(
            {
                "page_num": block["page_num"],
                "block_type": block["block_type"],
                "snippet": snippet or "；".join(first_sentences(block["text"], limit=2)),
            }
        )
        seen_pages.add(block["page_num"])
        if len(evidence) >= 4:
            break

    if not evidence:
        for page in ranked_pages[:3]:
            evidence.append(
                {
                    "page_num": page["page_num"],
                    "block_type": "page_preview",
                    "snippet": "；".join(page["preview"]),
                }
            )

    cited_pages = sorted({item["page_num"] for item in evidence} or {page["page_num"] for page in ranked_pages})
    summary_line = ""
    if evidence:
        summary_line = f"核心结论：{evidence[0]['snippet']}"
    bullet_lines = [f"- 第{item['page_num']}页：{item['snippet']}" for item in evidence[:4]]
    answer = (
        "基于检索到的财报页面，整理出的回答如下：\n"
        + (summary_line + "\n" if summary_line else "")
        + "\n".join(bullet_lines)
        + "\n引用页码："
        + "、".join(f"第{page}页" for page in cited_pages)
    )

    citations = [
        {
            "page_num": page["page_num"],
            "image_path": page["image_path"],
            "preview": page["preview"],
        }
        for page in ranked_pages
    ]
    return {
        "query": query,
        "answer": answer,
        "citations": citations,
        "evidence": evidence,
    }


def run_query(query: str) -> dict:
    pages, blocks = load_index()
    ranked_pages, ranked_blocks = rank_pages(query, pages, blocks)
    return synthesize_answer(query, ranked_pages, ranked_blocks)


def interactive_chat() -> None:
    print("=" * 50)
    print("多模态财报助手（离线页级引用版）")
    print("输入 exit 退出")
    print("=" * 50)
    while True:
        query = input("\n>>> 请提问: ").strip()
        if query.lower() in {"exit", "quit"}:
            break
        if not query:
            continue
        result = run_query(query)
        print("\n" + result["answer"])
        print("\n引用详情：")
        for citation in result["citations"]:
            print(f"- 第{citation['page_num']}页 | {citation['image_path']}")


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument("--query", type=str, default="")
    args = parser.parse_args()
    if args.query:
        print(json.dumps(run_query(args.query), ensure_ascii=False, indent=2))
    else:
        interactive_chat()


if __name__ == "__main__":
    main()
